In [ ]:
import time
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

BASE_URL = 'https://lifehacker.ru'
LIST_URL = f'{BASE_URL}/topics/technology/'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}

### Поиск элементов в разметке

В инструментах разработчика (F12) на странице материала найдены:
- **Заголовок**: тег h1 (на странице статьи он один).
- **Текст материала**: тег article с уникальным классом single-article__post-content; весь текст лежит в абзацах `<p>` внутри него.

На странице списка ссылки на материалы лежат в тегах <a> с атрибутом title, у которого значение равно заголовку материала. Служебные ссылки (логотип, счётчик комментариев) отфильтрованы по тексту title.

### Пагинация
Страницы рубрики «Технологии» имеют вид `https://lifehacker.ru/topics/technology/?page=N`, где N это номер страницы (первая страница открывается и без параметра).

In [8]:
TEXT_SELECTOR = 'article.single-article__post-content'

def parse_article(html):
    soup = BeautifulSoup(html, 'lxml')

    h1 = soup.find('h1')
    title = h1.get_text(strip=True) if h1 else None

    body = soup.select_one(TEXT_SELECTOR)
    if body is not None:
        text = ' '.join(p.get_text(' ', strip=True) for p in body.find_all('p'))
        text = text.replace('\xa0', ' ')
    else:
        text = None
    return title, text

result = []
for url, html in pages_html.items():
    title, text = parse_article(html)
    result.append({'url': url, 'title': title, 'text': text})

In [ ]:
pages_html = {}

for url in tqdm(article_urls):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        pages_html[url] = resp.text
    except requests.RequestException as e:
        print('Ошибка:', url, e)
    time.sleep(0.5)

print(len(pages_html))

100%|██████████| 300/300 [10:39<00:00,  2.13s/it]

300


In [ ]:
TEXT_SELECTOR = 'article.single-article__post-content'

def parse_article(html):
    soup = BeautifulSoup(html, 'lxml')

    h1 = soup.find('h1')
    title = h1.get_text(strip=True) if h1 else None

    body = soup.select_one(TEXT_SELECTOR)
    if body is not None:
        text = ' '.join(p.get_text(' ', strip=True) for p in body.find_all('p'))
    else:
        text = None
    return title, text

result = []
for url, html in pages_html.items():
    title, text = parse_article(html)
    result.append({'url': url, 'title': title, 'text': text})

In [ ]:
test_url = article_urls[0]
print(parse_article(pages_html[test_url]))

In [ ]:
pd.set_option('display.max_colwidth', 200)

data = pd.DataFrame(result)
print(data.shape)
print(data.isna().sum())
data.head()